# Agent Configuration Quality Validator

Validate your Fabric Data Agent configuration against Microsoft best practices.

**Reference**: [Best practices for configuring your data agent](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-configuration-best-practices)

## How to Use
1. **Configure your agent** in the input section below
2. **Run all cells** to execute validation checks  
3. **Review results** and follow recommendations for improvements

## Validation Checks
✅ **Benchmark Coverage** - Test cases cover all business domains  
✅ **Instruction Quality** - Agent and data source instructions completeness  
✅ **Example Query Quality** - Few-shot examples follow best practices  
✅ **Join Risk Assessment** - Identifies cross-lakehouse join complexity  
✅ **Failure Diagnosis** - Analyzes failure patterns with recommended fixes  
✅ **Iteration Planning** - Next improvement actions

In [ ]:
# Import required libraries
import pandas as pd
import json

## 📝 Step 1: Configure Your Agent

Replace the sample values below with your actual agent configuration.

In [ ]:
# ============================================================================
# INPUT YOUR AGENT CONFIGURATION HERE
# ============================================================================

# 1. Agent Instructions
#    Define: role, tone, data source routing, fallback behavior
comprehensive_agent_instructions = """
# REPLACE WITH YOUR AGENT INSTRUCTIONS

## Tone and Style
[How should the agent communicate?]

## General Knowledge
[What is the agent's purpose and domain?]

## Data Source Routing  
- When asked about [topic] → Use [lakehouse.table]

## Fallback Behavior
[What happens when data is missing or unclear?]
"""

# 2. Data Source Instructions
#    Define: schemas, joins, filters, example values
comprehensive_data_source_instructions = """
# REPLACE WITH YOUR DATA SOURCE INSTRUCTIONS

## Table Relationships
[Document table joins and foreign keys]

## Required Filters  
[Specify mandatory WHERE clauses]

## Example Values
[Provide sample values for common columns]
"""

# 3. Few-Shot Examples
#    Provide: question variations and validated SQL
few_shot_examples = [
    {
        "question": "What are total sales by category?",
        "sql": "SELECT category, SUM(sales_amount) FROM lakehouse.sales GROUP BY category"
    },
    # Add more examples...
]

# 4. Join Relationships
#    Document: all table joins the agent will execute
join_relationships = pd.DataFrame(
    columns=["left_table", "right_table", "join_key", "join_type", "cardinality"],
    data=[
        ["lakehouse1.sales_fact", "lakehouse1.product_dim", "product_id", "INNER", "many-to-one"],
        # Add more joins...
    ]
)

print("✓ Configuration loaded")
print(f"  • Few-shot examples: {len(few_shot_examples)}")
print(f"  • Join relationships: {len(join_relationships)}")

## 📊 Step 2: Run Validation Checks

### Check 1: Benchmark Coverage

Validates that test cases cover all business domains and complexity levels.

In [ ]:
# Optional: Define your benchmark test set (sample ground-truth cases)
# This helps track coverage across business domains

benchmark_df = pd.DataFrame(
    columns=["question", "expected_query", "expected_answer", "domain", "complexity", "notes"],
    data=[
        # Add your benchmark test cases here
        # Format: [question, expected_sql, expected_answer, domain (sales/customer/product), complexity (low/medium/high), notes]
    ]
)

def benchmark_coverage_report(df):
    if df.empty:
        return {"status": "warning", "message": "No benchmark cases defined (optional)", "total_cases": 0}
    
    required_domains = {"sales", "customer", "product"}  # Adjust for your use case
    current_domains = set(df["domain"].dropna().str.lower())
    missing_domains = sorted(required_domains - current_domains)
    complexity_mix = df["complexity"].value_counts(dropna=False).to_dict()
    
    return {
        "status": "ok",
        "total_cases": len(df),
        "domains": sorted(current_domains),
        "missing_domains": missing_domains,
        "complexity_mix": complexity_mix,
        "recommendation": "Expand benchmark to cover missing domains" if missing_domains else "Good coverage"
    }

benchmark_report = benchmark_coverage_report(benchmark_df)
print("Benchmark Coverage Report")
print("=" * 60)
print(f"Status: {benchmark_report['status']}")
print(f"Total cases: {benchmark_report['total_cases']}")
if benchmark_report['total_cases'] > 0:
    print(f"Domains covered: {', '.join(benchmark_report['domains'])}")
    print(f"Missing domains: {', '.join(benchmark_report['missing_domains']) if benchmark_report['missing_domains'] else 'None'}")
    print(f"Complexity mix: {benchmark_report['complexity_mix']}")
print(f"Recommendation: {benchmark_report.get('recommendation', 'Define benchmark cases for iterative improvement')}")
if not benchmark_df.empty:
    display(benchmark_df)

### Check 2: Instruction Quality

Validates agent and data source instructions contain required guidance.

In [ ]:
def check_instruction_quality(agent_text, data_source_text):
    agent_lower = agent_text.lower()
    ds_lower = data_source_text.lower()
    
    # Check agent instructions
    required_agent = [
        ("when asked about", "Routing rules"),
        ("if data is missing", "Fallback behavior"),
        ("fully-qualified", "Table naming rules"),
        ("only answer", "Scope definition")
    ]
    
    # Check data source instructions
    required_ds = [
        ("join", "Join patterns"),
        ("example", "Example values"),
        ("key", "Key columns"),
        ("filter", "Required filters")
    ]
    
    missing_agent = [desc for phrase, desc in required_agent if phrase not in agent_lower]
    missing_ds = [desc for phrase, desc in required_ds if phrase not in ds_lower]
    
    # Check consistency
    consistency_issues = []
    if "fiscal" in agent_lower and "fiscal" not in ds_lower:
        consistency_issues.append("Agent mentions 'fiscal' but data source doesn't map to columns")
    if "delivered" in ds_lower and "delivered" not in agent_lower:
        consistency_issues.append("Data source has 'delivered' logic but agent doesn't enforce it")
    
    return {
        "agent_missing": missing_agent,
        "data_source_missing": missing_ds,
        "consistency_issues": consistency_issues,
        "status": "pass" if not missing_agent and not missing_ds and not consistency_issues else "needs_improvement"
    }

instruction_quality = check_instruction_quality(
    comprehensive_agent_instructions,
    comprehensive_data_source_instructions
)

print("Instruction Quality Check")
print("=" * 60)
print(f"Status: {instruction_quality['status'].upper()}")
print(f"\nAgent Instructions:")
if instruction_quality['agent_missing']:
    print(f"  ⚠ Missing: {', '.join(instruction_quality['agent_missing'])}")
else:
    print(f"  ✓ All required elements present")

print(f"\nData Source Instructions:")
if instruction_quality['data_source_missing']:
    print(f"  ⚠ Missing: {', '.join(instruction_quality['data_source_missing'])}")
else:
    print(f"  ✓ All required elements present")

if instruction_quality['consistency_issues']:
    print(f"\nConsistency Issues:")
    for issue in instruction_quality['consistency_issues']:
        print(f"  ⚠ {issue}")

### Check 3: Example Query Quality

Validates few-shot examples follow best practices.

In [ ]:
def assess_example_query_quality(examples):
    if not examples:
        return pd.DataFrame([{"status": "error", "message": "No few-shot examples provided"}])
    
    assessment_rows = []
    for example in examples:
        sql_text = example.get("sql", "").lower()
        question_text = example.get("question", "").lower()
        
        checks = {
            "fully_qualified": "lakehouse" in sql_text or "." in sql_text,
            "has_join_when_needed": (" join " in sql_text) or ("customer" not in question_text and "category" not in question_text),
            "has_filter": any(token in sql_text for token in ["where", "like", "between"]),
            "explicit_columns": "select" in sql_text and "*" not in sql_text,
        }
        score = sum(checks.values())
        
        assessment_rows.append({
            "question": example.get("question", "N/A")[:50],
            "score": f"{score}/4",
            "fully_qualified": "✓" if checks["fully_qualified"] else "✗",
            "has_join": "✓" if checks["has_join_when_needed"] else "✗",
            "has_filter": "✓" if checks["has_filter"] else "✗",
            "explicit_cols": "✓" if checks["explicit_columns"] else "✗",
            "status": "good" if score >= 3 else "needs work"
        })
    
    return pd.DataFrame(assessment_rows)

example_quality_df = assess_example_query_quality(few_shot_examples)

print("Example Query Quality Assessment")
print("=" * 60)
display(example_quality_df)

needs_improvement = example_quality_df[example_quality_df["status"] == "needs work"]
if not needs_improvement.empty:
    print(f"\n⚠ {len(needs_improvement)} example(s) need improvement")
else:
    print("\n✓ All examples meet quality standards")

### Check 4: Join Risk Assessment

Identifies potential cross-lakehouse join complexity.

In [ ]:
def assess_join_risk(join_df):
    if join_df.empty:
        return {"status": "warning", "message": "No join relationships defined", "risk_level": "unknown"}
    
    cross_lakehouse_count = sum(
        join_df["left_table"].str.split(".").str[0] != 
        join_df["right_table"].str.split(".").str[0]
    )
    
    duplicate_keys = join_df["join_key"].value_counts()
    duplicate_keys = duplicate_keys[duplicate_keys > 1].to_dict()
    
    risk_level = "high" if cross_lakehouse_count > 2 else "medium" if cross_lakehouse_count > 0 else "low"
    
    return {
        "status": "ok",
        "join_count": len(join_df),
        "cross_lakehouse_joins": int(cross_lakehouse_count),
        "duplicate_keys": duplicate_keys,
        "risk_level": risk_level,
        "recommendation": (
            "Add multiple cross-lakehouse examples and explicit join documentation" 
            if risk_level != "low" 
            else "Join structure is straightforward"
        )
    }

join_risk = assess_join_risk(join_relationships)

print("Join Risk Assessment")
print("=" * 60)
print(f"Risk Level: {join_risk.get('risk_level', 'unknown').upper()}")
if join_risk.get('status') == 'ok':
    print(f"Total joins: {join_risk['join_count']}")
    print(f"Cross-lakehouse joins: {join_risk['cross_lakehouse_joins']}")
    if join_risk['duplicate_keys']:
        print(f"Duplicate key usage: {join_risk['duplicate_keys']}")
print(f"\nRecommendation: {join_risk.get('recommendation', 'Define join relationships')}")

if not join_relationships.empty:
    print("\nJoin Relationships:")
    display(join_relationships)

### Check 5: Failure Diagnosis (Optional)

Analyzes known failure cases and suggests root causes.

In [ ]:
# Optional: Log known failures for diagnosis
# Format: question, generated_query, expected_behavior, issue_notes

failure_log = pd.DataFrame(
    columns=["question", "generated_query", "expected_behavior", "issue_notes"],
    data=[
        # Add failure cases here for automated diagnosis
        # Example:
        # ["Show top customers", "SELECT * FROM sales_fact", "Should join customer_dim", "Missing join"]
    ]
)

def diagnose_failure(question, generated_query, expected_behavior, issue_notes=""):
    query_lower = (generated_query or "").lower()
    expected_lower = (expected_behavior or "").lower()
    combined = " ".join([question.lower(), expected_lower, issue_notes.lower()])
    
    findings = []
    if "join" in combined and " join " not in query_lower:
        findings.append("Missing or incorrect join logic")
    if "lakehouse" in combined and "lakehouse" not in query_lower:
        findings.append("Missing fully-qualified table names")
    if "like" in combined and " like " not in query_lower:
        findings.append("Filter pattern should use LIKE")
    if any(token in combined for token in ["delivered", "active", "current"]) and "where" not in query_lower:
        findings.append("Missing required business-rule filter")
    
    if not findings:
        findings.append("Review instructions and examples manually")
    
    remediation_map = {
        "Missing or incorrect join logic": "Add join examples and explicit join instructions",
        "Missing fully-qualified table names": "Move naming rules to data source instructions",
        "Filter pattern should use LIKE": "Add LIKE pattern examples with leading words",
        "Missing required business-rule filter": "Document mandatory filters in instructions",
    }
    
    root_cause = findings[0]
    return {
        "question": question[:50],
        "root_cause": root_cause,
        "recommended_fix": remediation_map.get(root_cause, "Inspect and update configuration")
    }

if not failure_log.empty:
    diagnosis_results = pd.DataFrame([
        diagnose_failure(row.question, row.generated_query, row.expected_behavior, row.issue_notes)
        for row in failure_log.itertuples(index=False)
    ])
    
    print("Failure Diagnosis")
    print("=" * 60)
    display(diagnosis_results)
else:
    print("Failure Diagnosis")
    print("=" * 60)
    print("No failure cases logged (this is optional)")

### Check 6: Iteration Planning

Summarizes all findings and recommends next actions.

In [ ]:
iteration_actions = []

# Prioritize actions based on validation results
if benchmark_report.get("missing_domains", []):
    iteration_actions.append("Expand benchmark to cover missing business domains")

if instruction_quality["agent_missing"] or instruction_quality["data_source_missing"]:
    iteration_actions.append("Strengthen instructions before adding more examples")

if instruction_quality.get("consistency_issues", []):
    iteration_actions.append("Fix consistency issues between agent and data source instructions")

if join_risk.get("risk_level") in ["medium", "high"]:
    iteration_actions.append("Add more cross-lakehouse join examples and documentation")

if not example_quality_df.empty:
    needs_work = example_quality_df[example_quality_df["status"] == "needs work"]
    if not needs_work.empty:
        iteration_actions.append(f"Refine {len(needs_work)} low-scoring few-shot examples")

if not iteration_actions:
    iteration_actions.append("Configuration looks good! Test with real queries and monitor failures")

iteration_plan = pd.DataFrame({
    "priority": list(range(1, len(iteration_actions) + 1)),
    "next_action": iteration_actions
})

print("=" * 60)
print("ITERATION PLAN - Recommended Next Actions")
print("=" * 60)
display(iteration_plan)

print("\n" + "=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)

## 📚 Additional Resources

- [Microsoft Learn: Best practices for configuring your data agent](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-configuration-best-practices)
- [Microsoft Learn: Adopt an iterative process to improve your data agent](https://learn.microsoft.com/en-us/fabric/data-science/data-agent-iterative-improvement)
- [Fabric Data Agent SDK](https://pypi.org/project/fabric-data-agent-sdk/)